# La pire image

In [1]:
import tkinter as tk
import time

FRAME_COUNT = 1000
BUDGET_MS = 11.0

root = tk.Tk()
root.geometry("800x600")
canvas = tk.Canvas(root, width=800, height=600, bg="black")
canvas.pack()

frame_times = []

def run_test():
    for i in range(FRAME_COUNT):
        start = time.perf_counter()

        canvas.delete("all")
        canvas.create_rectangle(0, 0, 800, 600, fill="#1a1a26")
        canvas.update()

        end = time.perf_counter()
        ms = (end - start) * 1000
        frame_times.append(ms)

    max_ms = max(frame_times)
    over_budget = sum(1 for t in frame_times if t > BUDGET_MS)

    print(f"Image la plus longue : {max_ms:.3f} ms")
    print(f"Images au-dessus de {BUDGET_MS:.0f} ms : {over_budget} / {FRAME_COUNT}")

    root.destroy()

root.after(100, run_test)
root.mainloop()

Image la plus longue : 10.438 ms
Images au-dessus de 11 ms : 0 / 1000


# Le cout du doublement

In [2]:
FRAME_COUNT = 1000
BUDGET_MS = 11.0

root = tk.Tk()
root.geometry("800x600")
canvas = tk.Canvas(root, width=800, height=600, bg="black")
canvas.pack()

render_times = []

def run_test():
    for i in range(FRAME_COUNT):
        start = time.perf_counter()

       
        canvas.delete("all")
        canvas.create_rectangle(0, 0, 800, 600, fill="#1a1a26")
        canvas.update()
      

        end = time.perf_counter()
        ms = (end - start) * 1000
        render_times.append(ms)

    avg_ms = sum(render_times) / len(render_times)
    max_ms = max(render_times)

    print(f"Rendu moyen (une image) : {avg_ms:.3f} ms")
    print(f"Rendu maximal (une image) : {max_ms:.3f} ms")
    print(f"Rendu estimé x2 (moyenne) : {avg_ms * 2:.3f} ms")
    print(f"Rendu estimé x2 (max) : {max_ms * 2:.3f} ms")
    print(f"Reste pour le reste (moyenne, sur budget {BUDGET_MS} ms) : {BUDGET_MS - avg_ms * 2:.3f} ms")
    print(f"Reste pour le reste (max, sur budget {BUDGET_MS} ms) : {BUDGET_MS - max_ms * 2:.3f} ms")

    root.destroy()

root.after(100, run_test)
root.mainloop()

Rendu moyen (une image) : 1.312 ms
Rendu maximal (une image) : 4.119 ms
Rendu estimé x2 (moyenne) : 2.623 ms
Rendu estimé x2 (max) : 8.238 ms
Reste pour le reste (moyenne, sur budget 11.0 ms) : 8.377 ms
Reste pour le reste (max, sur budget 11.0 ms) : 2.762 ms


# Vingt millisecondes senties

In [10]:
import tkinter as tk
import time
from collections import deque

WIDTH, HEIGHT = 900, 600

root = tk.Tk()
root.title("Retard réglable - exercice 12")
root.geometry(f"{WIDTH}x{HEIGHT}")

canvas = tk.Canvas(root, width=WIDTH, height=HEIGHT, bg="#1a1a26")
canvas.pack()

history = deque(maxlen=2000)

delay_var = tk.IntVar(value=25)

slider = tk.Scale(
    root, from_=0, to=200, orient="horizontal",
    variable=delay_var, label="Retard (ms) - déplace ce curseur pendant le test",
    length=700, resolution=5
)
slider.pack(pady=5)

label = tk.Label(root, text="Bouge la souris dans la zone. Le point rouge suit avec le retard réglé ci-dessus.")
label.pack()

dot = canvas.create_oval(0, 0, 20, 20, fill="red", outline="")

def on_motion(event):
    now = time.perf_counter()
    history.append((now, event.x, event.y))

canvas.bind("<Motion>", on_motion)

def update_dot():
    now = time.perf_counter()
    delay_s = delay_var.get() / 1000.0
    target_time = now - delay_s

    chosen = None
    for t, x, y in reversed(history):
        if t <= target_time:
            chosen = (x, y)
            break
    if chosen is None and history:
        chosen = (history[0][1], history[0][2])

    if chosen:
        x, y = chosen
        canvas.coords(dot, x - 10, y - 10, x + 10, y + 10)

    root.after(8, update_dot)  

update_dot()
root.mainloop()